# Food-101 Colab GPU Training Workflow

This notebook uses Colab as temporary GPU compute. GitHub provides the repository source, `/content` stores temporary data and training outputs, and Google Drive is used only when explicitly archiving selected final runs.

The final report remains `food101_CNN_final_project.ipynb`.

## 1. Runtime Setup

Use `Runtime -> Change runtime type -> GPU` in Colab. The setup cell clones the configured GitHub branch into `/content/food101-cnn`, installs the package, and creates temporary runtime folders.

In [1]:
from __future__ import annotations

import csv
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
GITHUB_REPO_URL = "https://github.com/alexal00/food101-cnn.git"
GITHUB_BRANCH = "codex/final-deliverable-pr"  # Change only when testing another branch.
CLONE_FRESH = False  # Preserve an existing checkout and local runtime data cache when rerunning cells.
RUN_INSTALL = True

SPARSE_CLONE_WITHOUT_DATA = False  # Keep the versioned processed cache in /content for fast local training reads.
USE_DRIVE_DATA_SOURCE = True  # Use Drive only as a source/archive fallback, never as the live training data root.
DRIVE_DATA_ROOT = Path("/content/drive/MyDrive/food101-cnn/data")
DRIVE_DATA_ARCHIVE = Path("/content/drive/MyDrive/food101-cnn/food101-data.tar.zst")
EXTRACT_DATA_ARCHIVE_IF_NEEDED = True
PULL_REPO_LFS_DATA = False  # Set True only when training from data stored in Git LFS.

CONTENT_ROOT = Path("/content") if IN_COLAB else Path.cwd().resolve()
PROJECT_ROOT = CONTENT_ROOT / "food101-cnn" if IN_COLAB else Path.cwd().resolve()
RUN_ROOT = CONTENT_ROOT / "food101-runs" if IN_COLAB else PROJECT_ROOT / "outputs" / "runs"
DRIVE_FINAL_RUN_ROOT = Path("/content/drive/MyDrive/food101-cnn/final-runs")
DATA_ROOT = PROJECT_ROOT / "data"


def run_setup_command(
    command: list[str],
    *,
    cwd: Path | None = None,
    env: dict[str, str] | None = None,
) -> None:
    print("$", " ".join(command))
    command_env = os.environ.copy()
    if env:
        command_env.update(env)
    subprocess.run(command, cwd=cwd, env=command_env, check=True)


def clone_repository() -> None:
    command = ["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH]
    env = {"GIT_LFS_SKIP_SMUDGE": "1"} if not PULL_REPO_LFS_DATA else None
    if SPARSE_CLONE_WITHOUT_DATA:
        command.extend(["--filter=blob:none", "--sparse"])
    command.extend([GITHUB_REPO_URL, str(PROJECT_ROOT)])
    run_setup_command(command, env=env)
    if SPARSE_CLONE_WITHOUT_DATA:
        configure_sparse_checkout_without_data()


def configure_sparse_checkout_without_data() -> None:
    sparse_file = PROJECT_ROOT / ".git" / "info" / "sparse-checkout"
    sparse_file.parent.mkdir(parents=True, exist_ok=True)
    sparse_file.write_text("/*\n!/data/\n!/data/**\n", encoding="utf-8")
    run_setup_command(["git", "config", "core.sparseCheckout", "true"], cwd=PROJECT_ROOT)
    run_setup_command(["git", "config", "core.sparseCheckoutCone", "false"], cwd=PROJECT_ROOT)
    run_setup_command(["git", "read-tree", "-mu", "HEAD"], cwd=PROJECT_ROOT)


def restore_full_checkout() -> None:
    if not (PROJECT_ROOT / ".git").is_dir():
        return
    try:
        run_setup_command(["git", "sparse-checkout", "disable"], cwd=PROJECT_ROOT)
    except subprocess.CalledProcessError:
        run_setup_command(["git", "config", "core.sparseCheckout", "false"], cwd=PROJECT_ROOT)
        run_setup_command(["git", "read-tree", "-mu", "HEAD"], cwd=PROJECT_ROOT)


def check_git_lfs_available(*, required: bool) -> bool:
    try:
        run_setup_command(["git", "lfs", "version"], cwd=PROJECT_ROOT)
    except (FileNotFoundError, subprocess.CalledProcessError) as exc:
        message = (
            "Git LFS is unavailable in this Colab runtime. "
            "Install git-lfs before pulling repository-managed dataset files."
        )
        if required:
            raise RuntimeError(message) from exc
        print(f"{message} Continuing because repository LFS data is not being pulled.")
        return False
    run_setup_command(["git", "lfs", "install", "--local"], cwd=PROJECT_ROOT)
    return True


def pull_git_lfs_objects() -> None:
    run_setup_command(["git", "lfs", "pull"], cwd=PROJECT_ROOT)


def ensure_local_data_root(data_root: Path) -> Path:
    if data_root.is_symlink():
        previous_target = data_root.resolve()
        data_root.unlink()
        print(f"Detached data symlink: {data_root} (was {previous_target})")
    data_root.mkdir(parents=True, exist_ok=True)
    return data_root


def copy_drive_data_cache_if_needed(source_data_root: Path, data_root: Path) -> None:
    if not USE_DRIVE_DATA_SOURCE or not IN_COLAB:
        return
    if not data_archive_needed(data_root):
        print(f"Using local runtime data cache under: {data_root}")
        return
    if not source_data_root.exists():
        print(f"Drive data source not found: {source_data_root}")
        return

    for relative_dir in [Path("processed"), Path("reports")]:
        source = source_data_root / relative_dir
        destination = data_root / relative_dir
        if source.exists():
            print(f"Copying Drive data source: {source} -> {destination}")
            shutil.copytree(source, destination, dirs_exist_ok=True)


def latest_cached_index(cache_root: Path) -> Path | None:
    candidates = sorted(
        cache_root.glob("*/cached_index.csv"),
        key=lambda candidate: candidate.stat().st_mtime,
        reverse=True,
    )
    return candidates[0] if candidates else None


def cached_image_header_is_readable(path: Path) -> bool:
    if not path.is_file() or path.stat().st_size == 0:
        return False
    header = path.read_bytes()[:64]
    if header.startswith(b"version https://git-lfs.github.com/spec/v1"):
        return False
    return header.startswith((b"\xff\xd8\xff", b"\x89PNG\r\n\x1a\n", b"RIFF"))


def processed_cache_ready(cache_root: Path) -> tuple[bool, Path | None]:
    cached_index = latest_cached_index(cache_root)
    if cached_index is None:
        return False, None
    images_dir = cached_index.parent / "images"
    if not images_dir.is_dir() or cached_index.stat().st_size == 0:
        return False, cached_index
    with cached_index.open("r", newline="", encoding="utf-8") as handle:
        first_row = next(csv.DictReader(handle), None)
    if not first_row or not first_row.get("relative_path"):
        return False, cached_index
    first_image = images_dir / first_row["relative_path"]
    return cached_image_header_is_readable(first_image), cached_index

def data_archive_needed(data_root: Path) -> bool:
    ready, _ = processed_cache_ready(data_root / "processed")
    return not ready


def ensure_zstd_available() -> None:
    if shutil.which("zstd"):
        return
    run_setup_command(["apt-get", "update"])
    run_setup_command(["apt-get", "install", "-y", "zstd"])


def extract_zstd_tar(archive_path: Path, extract_root: Path) -> None:
    print("$", f"zstd -dc {archive_path} | tar -xf - -C {extract_root}")
    zstd_process = subprocess.Popen(
        ["zstd", "-dc", str(archive_path)],
        stdout=subprocess.PIPE,
    )
    assert zstd_process.stdout is not None
    try:
        subprocess.run(
            ["tar", "-xf", "-", "-C", str(extract_root)],
            stdin=zstd_process.stdout,
            check=True,
        )
    finally:
        zstd_process.stdout.close()
    returncode = zstd_process.wait()
    if returncode != 0:
        raise subprocess.CalledProcessError(returncode, ["zstd", "-dc", str(archive_path)])


def extract_data_archive_if_needed(archive_path: Path, data_root: Path) -> None:
    if not EXTRACT_DATA_ARCHIVE_IF_NEEDED:
        return
    if not data_archive_needed(data_root):
        print(f"Local runtime data cache is ready under: {data_root}")
        return
    if not archive_path.is_file():
        print(f"Data archive not found: {archive_path}")
        print("Processed cache fallback will download and rebuild data if needed.")
        return

    ensure_zstd_available()
    extract_root = data_root.resolve().parent
    print(f"Extracting data archive into local runtime storage: {archive_path} -> {extract_root}")
    extract_zstd_tar(archive_path, extract_root)


if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    if PROJECT_ROOT.exists() and CLONE_FRESH:
        shutil.rmtree(PROJECT_ROOT)
    if not PROJECT_ROOT.exists():
        clone_repository()
    elif not SPARSE_CLONE_WITHOUT_DATA:
        restore_full_checkout()
else:
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / "pyproject.toml").is_file():
            PROJECT_ROOT = candidate
            RUN_ROOT = PROJECT_ROOT / "outputs" / "runs"
            DATA_ROOT = PROJECT_ROOT / "data"
            break

os.chdir(PROJECT_ROOT)
if IN_COLAB:
    lfs_available = check_git_lfs_available(required=PULL_REPO_LFS_DATA)
    run_setup_command(["git", "fetch", "origin", GITHUB_BRANCH], cwd=PROJECT_ROOT)
    run_setup_command(
        ["git", "checkout", GITHUB_BRANCH],
        cwd=PROJECT_ROOT,
        env={"GIT_LFS_SKIP_SMUDGE": "1"} if not PULL_REPO_LFS_DATA else None,
    )
    if SPARSE_CLONE_WITHOUT_DATA:
        configure_sparse_checkout_without_data()
    else:
        restore_full_checkout()
    run_setup_command(
        ["git", "pull", "--ff-only", "origin", GITHUB_BRANCH],
        cwd=PROJECT_ROOT,
        env={"GIT_LFS_SKIP_SMUDGE": "1"} if not PULL_REPO_LFS_DATA else None,
    )
    if SPARSE_CLONE_WITHOUT_DATA:
        configure_sparse_checkout_without_data()
    else:
        restore_full_checkout()
    DATA_ROOT = ensure_local_data_root(DATA_ROOT)
    copy_drive_data_cache_if_needed(DRIVE_DATA_ROOT, DATA_ROOT)
    extract_data_archive_if_needed(DRIVE_DATA_ARCHIVE, DATA_ROOT)
    if PULL_REPO_LFS_DATA and lfs_available:
        pull_git_lfs_objects()

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

pythonpath_parts = [str(SRC_ROOT), *(part for part in os.environ.get("PYTHONPATH", "").split(os.pathsep) if part)]
os.environ["PYTHONPATH"] = os.pathsep.join(dict.fromkeys(pythonpath_parts))

for directory in [DATA_ROOT / "raw", DATA_ROOT / "processed", DATA_ROOT / "reports", RUN_ROOT]:
    directory.mkdir(parents=True, exist_ok=True)
if IN_COLAB:
    DRIVE_FINAL_RUN_ROOT.mkdir(parents=True, exist_ok=True)

if RUN_INSTALL:
    run_setup_command([sys.executable, "-m", "pip", "install", "-e", ".[dev]"], cwd=PROJECT_ROOT)

required = [
    "pyproject.toml",
    "src/food101_cnn/config.py",
    "scripts/run_experiment_plan.py",
    "scripts/train.py",
    "scripts/evaluate.py",
    "scripts/export_model.py",
    "configs/baseline_cnn_simple.yaml",
    "configs/efficientnet_b0_gpu.yaml",
]
missing = [item for item in required if not (PROJECT_ROOT / item).exists()]
if missing:
    raise FileNotFoundError(f"Repository checkout is missing required files: {missing}")

print(json.dumps({
    "in_colab": IN_COLAB,
    "github_repo": GITHUB_REPO_URL,
    "github_branch": GITHUB_BRANCH,
    "project_root": str(PROJECT_ROOT),
    "data_root": str(DATA_ROOT),
    "drive_data_source_root": str(DRIVE_DATA_ROOT) if IN_COLAB and USE_DRIVE_DATA_SOURCE else None,
    "drive_data_archive": str(DRIVE_DATA_ARCHIVE) if IN_COLAB and USE_DRIVE_DATA_SOURCE else None,
    "extract_data_archive_if_needed": EXTRACT_DATA_ARCHIVE_IF_NEEDED if IN_COLAB else None,
    "sparse_clone_without_data": SPARSE_CLONE_WITHOUT_DATA if IN_COLAB else None,
    "pull_repo_lfs_data": PULL_REPO_LFS_DATA if IN_COLAB else None,
    "run_root": str(RUN_ROOT),
    "drive_final_run_root": str(DRIVE_FINAL_RUN_ROOT) if IN_COLAB else None,
    "pythonpath_first": os.environ["PYTHONPATH"].split(os.pathsep)[0],
}, indent=2))


Mounted at /content/drive
$ git clone --depth 1 --branch codex/final-deliverable-pr --filter=blob:none --sparse https://github.com/alexal00/food101-cnn.git /content/food101-cnn
$ git config core.sparseCheckout true
$ git config core.sparseCheckoutCone false
$ git read-tree -mu HEAD
$ git lfs version
$ git lfs install --local
$ git fetch origin codex/final-deliverable-pr
$ git checkout codex/final-deliverable-pr
$ git config core.sparseCheckout true
$ git config core.sparseCheckoutCone false
$ git read-tree -mu HEAD
$ git pull --ff-only origin codex/final-deliverable-pr
$ git config core.sparseCheckout true
$ git config core.sparseCheckoutCone false
$ git read-tree -mu HEAD
Linked Drive data root: /content/food101-cnn/data -> /content/drive/MyDrive/food101-cnn/data
$ apt-get update
$ apt-get install -y zstd
Extracting data archive: /content/drive/MyDrive/food101-cnn/food101-data.tar.zst -> /content/drive/MyDrive/food101-cnn
$ zstd -dc /content/drive/MyDrive/food101-cnn/food101-data.tar.

## 2. Environment Check

This verifies the installed package and available accelerator before any long-running command is launched.

In [2]:
import torch
import food101_cnn


def gpu_info() -> dict:
    if not torch.cuda.is_available():
        return {"device": "cpu", "name": "CPU", "memory_gb": 0.0}
    props = torch.cuda.get_device_properties(0)
    return {
        "device": "cuda",
        "name": props.name,
        "memory_gb": round(props.total_memory / 1024**3, 2),
    }


GPU = gpu_info()
print(json.dumps({
    "food101_cnn_version": food101_cnn.__version__,
    "module_path": food101_cnn.__file__,
    "gpu": GPU,
}, indent=2))

{
  "food101_cnn_version": "0.1.0",
  "module_path": "/content/food101-cnn/src/food101_cnn/__init__.py",
  "gpu": {
    "device": "cuda",
    "name": "NVIDIA A100-SXM4-40GB",
    "memory_gb": 39.49
  }
}


## 3. Execution Flags

Default values print the selected plan instead of launching expensive jobs. Data and run products stay in the runtime filesystem unless final archiving is enabled.

In [3]:
import csv

FORCE_DATA_PREP = False
FORCE_IMAGE_CACHE = False
RUN_TRAINING = True
RUN_EVALUATION = True
RUN_EXPORT_PT = True
RUN_ARCHIVE_FINAL_RUNS = True

PLAN_NAME = "gpu-default"  # Use "gpu-full" for the full comparison set.
PLAN_MODELS: list[str] = ["baseline_cnn_simple", "baseline_cnn_local"]  # Optional subset, for example ["efficientnet_b0"].
NUM_WORKERS = 8 if IN_COLAB else None  # Keeps GPU runs fed from local /content storage.
DATA_CONFIG = "configs/efficientnet_b0.yaml"

INDEX_CSV = DATA_ROOT / "reports" / "dataset_index.csv"
PROCESSED_ROOT = DATA_ROOT / "processed"


def latest_cached_index(cache_root: Path) -> Path | None:
    candidates = sorted(
        cache_root.glob("*/cached_index.csv"),
        key=lambda candidate: candidate.stat().st_mtime,
        reverse=True,
    )
    return candidates[0] if candidates else None


def processed_cache_ready(cache_root: Path) -> tuple[bool, Path | None]:
    cached_index = latest_cached_index(cache_root)
    if cached_index is None:
        return False, None
    images_dir = cached_index.parent / "images"
    if not images_dir.is_dir() or cached_index.stat().st_size == 0:
        return False, cached_index
    with cached_index.open("r", newline="", encoding="utf-8") as handle:
        first_row = next(csv.DictReader(handle), None)
    if not first_row or not first_row.get("relative_path"):
        return False, cached_index
    first_image = images_dir / first_row["relative_path"]
    return cached_image_header_is_readable(first_image), cached_index


LOCAL_PROCESSED_CACHE_READY, ACTIVE_CACHED_INDEX = processed_cache_ready(PROCESSED_ROOT)
RUN_DATA_PREP = FORCE_DATA_PREP or not LOCAL_PROCESSED_CACHE_READY
RUN_IMAGE_CACHE = FORCE_IMAGE_CACHE or not LOCAL_PROCESSED_CACHE_READY
if LOCAL_PROCESSED_CACHE_READY:
    print(f"Using processed image cache: {ACTIVE_CACHED_INDEX}")
else:
    print("Processed image cache is missing or invalid.")
    print(f"Expected processed cache root: {PROCESSED_ROOT}")
    print("Downloading Food-101 and rebuilding the processed image cache before training.")

RESUME_CHECKPOINTS = {
    # "resnet50": "/content/food101-runs/<run>/checkpoints/<run>_best_model.pt",
}

FINAL_RUN_MODELS = [
    # "efficientnet_b0",
]

print(json.dumps({
    "plan": PLAN_NAME,
    "models": PLAN_MODELS or "all plan models",
    "num_workers": NUM_WORKERS,
    "data_root": str(DATA_ROOT),
    "index_csv": str(INDEX_CSV),
    "processed_root": str(PROCESSED_ROOT),
    "processed_cache_ready": LOCAL_PROCESSED_CACHE_READY,
    "active_cached_index": str(ACTIVE_CACHED_INDEX) if ACTIVE_CACHED_INDEX else None,
    "run_data_prep": RUN_DATA_PREP,
    "run_image_cache": RUN_IMAGE_CACHE,
    "run_root": str(RUN_ROOT),
    "archive_enabled": RUN_ARCHIVE_FINAL_RUNS,
}, indent=2))


Using processed image cache: /content/food101-cnn/data/processed/f8e106c00724/cached_index.csv
{
  "plan": "gpu-default",
  "models": [
    "baseline_cnn_simple",
    "baseline_cnn_local"
  ],
  "data_root": "/content/food101-cnn/data",
  "index_csv": "/content/food101-cnn/data/reports/dataset_index.csv",
  "processed_root": "/content/food101-cnn/data/processed",
  "processed_cache_ready": true,
  "active_cached_index": "/content/food101-cnn/data/processed/f8e106c00724/cached_index.csv",
  "run_data_prep": false,
  "run_image_cache": false,
  "run_root": "/content/food101-runs",
  "archive_enabled": true
}


## 4. Plan Runner

The training plan is executed through `scripts/run_experiment_plan.py`, which delegates to `scripts/download_data.py`, `scripts/cache_images.py`, `scripts/train.py`, `scripts/evaluate.py`, and `scripts/export_model.py`. In Colab, the notebook keeps `PROJECT_ROOT/data` on `/content` local runtime storage. The versioned processed cache is used directly from the repository when present; Drive is only a source/archive fallback that can be copied or extracted into `/content` before training. When `data/processed/*/cached_index.csv` is already present, the notebook trains directly from that processed cache and leaves download, indexing, and image caching disabled unless the force flags are enabled.


In [4]:
from food101_cnn.utils.runs import latest_run_for_model, list_run_manifests, manifest_to_comparison_row


def run_cli(command: list[str]) -> subprocess.CompletedProcess:
    print("$", " ".join(str(item) for item in command))
    process = subprocess.Popen(
        [str(item) for item in command],
        cwd=PROJECT_ROOT,
        env=os.environ.copy(),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    output_tail: list[str] = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        output_tail.append(line)
        output_tail = output_tail[-80:]
    returncode = process.wait()
    if returncode != 0:
        raise RuntimeError(
            f"Command failed with exit code {returncode}: {' '.join(str(item) for item in command)}\n"
            f"Last command output:\n{''.join(output_tail)}"
        )
    return subprocess.CompletedProcess(command, returncode)


def build_plan_command() -> list[str]:
    command = [
        sys.executable,
        "scripts/run_experiment_plan.py",
        "--plan",
        PLAN_NAME,
        "--data-config",
        DATA_CONFIG,
        "--index-csv",
        str(INDEX_CSV),
        "--cache-root",
        str(PROCESSED_ROOT),
        "--run-root",
        str(RUN_ROOT),
        "--gpu-memory-gb",
        str(GPU["memory_gb"]),
        "--device",
        "cuda" if torch.cuda.is_available() else "cpu",
    ]
    if NUM_WORKERS is not None:
        command.extend(["--num-workers", str(NUM_WORKERS)])
    for model_name in PLAN_MODELS:
        command.extend(["--model", model_name])
    for model_name, checkpoint in RESUME_CHECKPOINTS.items():
        command.extend(["--resume-checkpoint", f"{model_name}={checkpoint}"])
    if RUN_DATA_PREP:
        command.append("--data-prep")
    if RUN_IMAGE_CACHE:
        command.append("--cache-images")
    if RUN_TRAINING:
        command.append("--train")
    if RUN_EVALUATION:
        command.append("--evaluate")
    if RUN_EXPORT_PT:
        command.append("--export-pt")
    if not any([RUN_DATA_PREP, RUN_IMAGE_CACHE, RUN_TRAINING, RUN_EVALUATION, RUN_EXPORT_PT]):
        command.append("--dry-run")
    return command


## 5. Execute Plan

With all execution flags set to `False`, this cell prints the selected plan. By default, data preparation and image caching run only when the processed cache is missing; training, evaluation, and export remain controlled by the flags above.

In [ ]:
plan_command = build_plan_command()
run_cli(plan_command)

$ /usr/bin/python3 scripts/run_experiment_plan.py --plan gpu-default --data-config configs/efficientnet_b0.yaml --index-csv /content/food101-cnn/data/reports/dataset_index.csv --cache-root /content/food101-cnn/data/processed --run-root /content/food101-runs --gpu-memory-gb 39.49 --device cuda --model baseline_cnn_simple --model baseline_cnn_local --train --evaluate --export-pt
00:36:29 | INFO | numexpr.utils | NumExpr defaulting to 12 threads.
00:37:23 | INFO | __main__ | training_setup run=baseline_cnn_simple_20260611-0237_vfull1 config=configs/baseline_cnn_simple.yaml index_csv=/content/drive/MyDrive/food101-cnn/data/processed/f8e106c00724/cached_index.csv model=baseline_cnn_simple version=vfull1 train_samples=64438 val_samples=11312 batch_size=96 device=cuda epochs=30 lr=0.001
2026-06-11 00:37:24.094968: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computati

## 6. Final Run Archive

Only complete selected runs are copied to Drive. A run without `manifest.json` is treated as partial and is not archived.

In [ ]:
def archive_complete_run(model_name: str) -> Path | None:
    if not IN_COLAB:
        print("Drive archiving is only available in Colab.")
        return None
    manifest = latest_run_for_model(PROJECT_ROOT, model_name, run_root=RUN_ROOT)
    if not manifest:
        print(f"No complete run manifest found for {model_name}")
        return None
    run_name = manifest["run_name"]
    source = RUN_ROOT / run_name
    manifest_path = source / "manifest.json"
    if not manifest_path.is_file():
        print(f"Skipping partial run without manifest: {source}")
        return None
    destination = DRIVE_FINAL_RUN_ROOT / run_name
    shutil.copytree(source, destination, dirs_exist_ok=True)
    print(f"archived {source} -> {destination}")
    return destination


archived = []
if RUN_ARCHIVE_FINAL_RUNS:
    for model_name in FINAL_RUN_MODELS:
        archived_path = archive_complete_run(model_name)
        if archived_path is not None:
            archived.append(str(archived_path))
else:
    print("Final-run archiving disabled.")

archived

## 7. Run Manifest Snapshot

Use this quick table to confirm what the runtime produced. The final report notebook performs the project-level comparison.

In [ ]:
import pandas as pd

manifests = list_run_manifests(PROJECT_ROOT, run_root=RUN_ROOT)
rows = [manifest_to_comparison_row(item) for item in manifests]
pd.DataFrame(rows, columns=["model", "version", "trained_at", "checkpoint", "top1", "top5", "macro_f1", "ece", "latency_ms", "params", "notes"])